In [2]:
import pandas as pd
import requests
from zipfile import ZipFile
import datetime
from dateutil.relativedelta import relativedelta
import os
import json
from tqdm import tqdm

# ---- Config ----
SYMBOL = "BTCUSDT"
MONTHS_TO_DOWNLOAD = 1
OUTPUT_DIR = "BTCUSDT_TBT_1_month_data"
REQUEST_TIMEOUT = 60
DOWNLOAD_CHUNK_SIZE = 1024 * 1024       # 1 MB chunks while downloading the zip
PROCESS_CHUNK_ROWS = 2_000_000          # rows per processing chunk -> one parquet file each

os.makedirs(OUTPUT_DIR, exist_ok=True)

today = datetime.date.today()
months = [(today - relativedelta(months=i)).strftime("%Y-%m") for i in range(1, MONTHS_TO_DOWNLOAD + 1)]
months.reverse()

COLUMNS = [
    "trade_id",
    "price",
    "qty",
    "quoteQty",
    "time",
    "isBuyerMaker",
    "isBestMatch"
]


def checkpoint_path(year_month):
    return os.path.join(OUTPUT_DIR, f"{SYMBOL}_{year_month}_checkpoint.json")


def load_checkpoint(year_month):
    """Returns last completed chunk index and last trade_id saved, or defaults if none exists."""
    path = checkpoint_path(year_month)
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {"last_chunk_index": -1, "last_trade_id": None}


def save_checkpoint(year_month, chunk_index, last_trade_id):
    path = checkpoint_path(year_month)
    with open(path, "w") as f:
        json.dump({"last_chunk_index": chunk_index, "last_trade_id": last_trade_id}, f)


def download_zip_with_progress(url, dest_path):
    """Streams the zip in chunks, supports resuming a partial download via HTTP Range header."""
    resume_byte_pos = os.path.getsize(dest_path) if os.path.exists(dest_path) else 0

    head = requests.head(url, timeout=REQUEST_TIMEOUT)
    total_size = int(head.headers.get("content-length", 0))

    if resume_byte_pos >= total_size and total_size > 0:
        return  # already fully downloaded

    headers = {"Range": f"bytes={resume_byte_pos}-"} if resume_byte_pos else {}
    mode = "ab" if resume_byte_pos else "wb"

    response = requests.get(url, headers=headers, stream=True, timeout=REQUEST_TIMEOUT)
    with open(dest_path, mode) as f, tqdm(
        total=total_size, initial=resume_byte_pos, unit="B", unit_scale=True,
        desc=f"Downloading {os.path.basename(dest_path)}"
    ) as pbar:
        for chunk in response.iter_content(chunk_size=DOWNLOAD_CHUNK_SIZE):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))


def process_month(year_month):
    """Download the month's zip (resumable), then parse and save in row-chunks (resumable via checkpoint)."""
    zip_path = os.path.join(OUTPUT_DIR, f"{SYMBOL}_trades_{year_month}.zip")
    url = f"https://data.binance.vision/data/spot/monthly/trades/{SYMBOL}/{SYMBOL}-trades-{year_month}.zip"

    # Step 1: download (resumable)
    download_zip_with_progress(url, zip_path)

    # Step 2: parse CSV in chunks, resuming from last completed chunk
    checkpoint = load_checkpoint(year_month)
    start_chunk = checkpoint["last_chunk_index"] + 1

    with ZipFile(zip_path) as zf:
        csv_name = zf.namelist()[0]

        # detect header row once
        with zf.open(csv_name) as f:
            first_line = f.readline().decode().strip()
        has_header = not first_line.split(",")[0].isdigit()

        with zf.open(csv_name) as f:
            reader = pd.read_csv(
                f,
                header=0 if has_header else None,
                names=COLUMNS,
                chunksize=PROCESS_CHUNK_ROWS
            )

            with tqdm(desc=f"Processing {year_month}", unit="chunk") as pbar:
                for chunk_index, df in enumerate(reader):
                    if chunk_index < start_chunk:
                        continue  # already saved in a previous run

                    df["time"] = pd.to_datetime(df["time"], unit="ms")
                    for col in ["isBuyerMaker", "isBestMatch"]:
                        if df[col].dtype == object:
                            df[col] = df[col].map({"True": True, "False": False})
                        else:
                            df[col] = df[col].astype(bool)

                    chunk_file = os.path.join(
                        OUTPUT_DIR, f"{SYMBOL}_{year_month}_chunk_{chunk_index:05d}.parquet"
                    )
                    df.to_parquet(chunk_file, engine="pyarrow", compression="snappy")

                    last_trade_id = int(df["trade_id"].iloc[-1])
                    save_checkpoint(year_month, chunk_index, last_trade_id)

                    pbar.update(1)
                    pbar.set_postfix(last_trade_id=last_trade_id)

    return f"[{year_month}] done, resumed from chunk {start_chunk}"


for month in months:
    result = process_month(month)
    print(result)

print("\nAll tasks completed!")

Processing 2026-09: 43chunk [01:13,  1.70s/chunk, last_trade_id=6.73e+9]

[2026-09] done, resumed from chunk 0

All tasks completed!
